# 概览

在本 notebook 中，我们将讲解在 LIBERO 上运行实验所需的全部组件，以及一些常见用法，例如如何在代码库中定义你自己的算法与策略（policy）网络结构。

1. 为你的算法准备数据集
2. 编写你自己的算法
    - 从 `Sequential` 基类进行子类化
3. 编写你自己的模型
4. 编写你的训练循环
5. 可视化结果

## 1. 实验

In [ ]:
# 从 hydra 导入 compose / initialize：用于“加载 + 组合”配置文件（LIBERO 用 Hydra 管理配置）
from hydra import compose, initialize

# 导入 LIBERO 的工具：get_libero_path 用来定位数据/资源路径（基于安装目录）
# 注意：benchmark 这个名字这里虽然导入了，但本单元后面实际用的是 get_benchmark() 来创建基准套件
from libero.libero import benchmark, get_libero_path

# 导入 hydra 主包（后面会清理全局 Hydra 状态）
import hydra
# pprint：把配置字典更好看地打印出来，方便检查当前配置
import pprint
# os：用于设置环境变量、拼路径等
import os

# 禁用 tokenizer 的并行提示（有时会刷屏警告；不影响功能）
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# 指定 PyOpenGL 使用 EGL（常用于无显示器/服务器环境的离屏渲染）
os.environ['PYOPENGL_PLATFORM'] = 'egl'

# OmegaConf：Hydra 配置底层用它来表示；这里把配置转成 YAML 字符串再读回 python dict
from omegaconf import OmegaConf
# yaml：解析 YAML 文本为 Python 对象
import yaml
# EasyDict：让你可以用 cfg.xxx 这种点号方式访问字典字段（更直观）
from easydict import EasyDict

# get_benchmark：根据 benchmark 名称返回对应的基准类（例如 libero_object）
from libero.libero.benchmark import get_benchmark
# 数据集相关：get_dataset 读取单任务示范数据；SequenceVLDataset 把视觉+语言封装成序列数据集
from libero.lifelong.datasets import (GroupedTaskDataset, SequenceVLDataset, get_dataset)
# 常用工具：get_task_embs 生成语言/任务嵌入；safe_device 把模型放到 CPU/GPU；create_experiment_dir 创建实验目录
from libero.lifelong.utils import (get_task_embs, safe_device, create_experiment_dir)

# 如果你在同一个 Python 进程里反复 initialize Hydra，可能会报“Hydra 已初始化”；这里先清掉全局状态
hydra.core.global_hydra.GlobalHydra.instance().clear()

### load the default hydra config
# 初始化 Hydra：告诉它配置文件所在目录（相对 notebooks/ 的路径：../libero/configs）
initialize(config_path="../libero/configs")
# compose：按 config.yaml 入口文件把所有默认配置合成到一起
hydra_cfg = compose(config_name="config")
# 把 Hydra 配置对象转成 YAML 字符串（便于后续用 yaml.safe_load 解析成普通 dict）
yaml_config = OmegaConf.to_yaml(hydra_cfg)
# 解析 YAML -> dict，再包成 EasyDict：后面就能用 cfg.policy / cfg.train 这种写法访问
cfg = EasyDict(yaml.safe_load(yaml_config))

# 创建一个漂亮打印器（缩进 2 个空格），用来输出配置片段
pp = pprint.PrettyPrinter(indent=2)
# 打印 policy 配置，方便你确认当前用的模型/网络设置
pp.pprint(cfg.policy)

# prepare lifelong learning
# cfg.folder：LIBERO 数据集根目录（里面有 libero_object/libero_spatial 等）
cfg.folder = get_libero_path("datasets")
# cfg.bddl_folder：任务描述文件（BDDL）所在目录，用于构建环境/任务
cfg.bddl_folder = get_libero_path("bddl_files")
# cfg.init_states_folder：每个任务的初始化状态（init states）目录，用于复现/评估时重置环境
cfg.init_states_folder = get_libero_path("init_states")
# 评估时并行进程数（这里设成 1，最稳妥，方便调试）
cfg.eval.num_procs = 1
# 每次评估跑多少条 episode（这里设成 5，减少时间）
cfg.eval.n_eval = 5

# 训练 epoch 数（这里设为 25；注意下一行打印里写的是“reduced to 5”，那只是提示文本，未必一致）
cfg.train.n_epochs = 25

# 打印一条提示：告诉你这个示例为了更快把训练轮数缩小了（文字不影响实际 cfg.train.n_epochs）
pp.pprint(f"Note that the number of epochs used in this example is intentionally reduced to 5.")

# task_order：任务顺序索引（0..21），不同索引对应不同的任务排列顺序
task_order = cfg.data.task_order_index # can be from {0 .. 21}, default to 0, which is [task 0, 1, 2 ...]
# 选择基准套件（benchmark）：libero_object / libero_spatial / libero_goal / libero_10 等
cfg.benchmark_name = "libero_object" # can be from {"libero_spatial", "libero_object", "libero_goal", "libero_10"}
# 根据 benchmark_name 得到对应 Benchmark 类，并用 task_order 实例化（里面包含 n_tasks、语言描述、数据文件名等）
benchmark = get_benchmark(cfg.benchmark_name)(task_order)

# prepare datasets from the benchmark
# datasets：存放每个任务的训练数据集对象（后面会变成视觉+语言序列数据集）
datasets = []
# descriptions：存放每个任务的自然语言描述（用来生成 task embedding）
descriptions = []
# shape_meta：记录观测/动作维度等元信息（各任务通常一致，这里用第一个任务初始化）
shape_meta = None
# n_tasks：该 benchmark 中任务总数
n_tasks = benchmark.n_tasks

# 逐个任务加载示范数据（演示数据通常是 .hdf5）
for i in range(n_tasks):
    # 目前假设同一 benchmark 下所有任务的 shape_meta 相同，所以只在 i==0 时初始化观测工具
    task_i_dataset, shape_meta = get_dataset(
            dataset_path=os.path.join(cfg.folder, benchmark.get_task_demonstration(i)),  # 单任务示范数据文件路径
            obs_modality=cfg.data.obs.modality,  # 观测模态配置：例如 rgb / proprio 等
            initialize_obs_utils=(i==0),  # 只在第一个任务时初始化观测处理工具（节省重复开销）
            seq_len=cfg.data.seq_len,  # 序列长度：一次训练样本包含多少步历史
    )
    # 取出该任务的语言描述（例如“pick up ... and place ...”），用于生成任务/语言嵌入
    # add language to the vision dataset, hence we call vl_dataset
    descriptions.append(benchmark.get_task(i).language)
    # 把当前任务的数据集对象收集起来
    datasets.append(task_i_dataset)

# 根据所有任务的语言描述生成 task embeddings（通常来自语言编码器，如 BERT/CLIP 文本等）
task_embs = get_task_embs(cfg, descriptions)
# 把 embeddings 写回 benchmark：后面评估/推理时可以按 task_id 取对应 embedding
benchmark.set_task_embs(task_embs)

# 把每个任务的数据集包装成 SequenceVLDataset：让每个样本同时带上视觉序列 + 语言 embedding
datasets = [SequenceVLDataset(ds, emb) for (ds, emb) in zip(datasets, task_embs)]
# 统计每个任务有多少条 demonstration（示范轨迹）
n_demos = [data.n_demos for data in datasets]
# 统计每个任务一共切成多少个训练序列（取决于轨迹长度与 seq_len）
n_sequences = [data.total_num_sequences for data in datasets]

/tmp/ipykernel_24770/2563292850.py:18: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../libero/configs")


{ 'color_aug': { 'network': 'BatchWiseImgColorJitterAug',
                 'network_kwargs': { 'brightness': 0.3,
                                     'contrast': 0.3,
                                     'epsilon': 0.1,
                                     'hue': 0.3,
                                     'input_shape': None,
                                     'saturation': 0.3}},
  'embed_size': 64,
  'extra_hidden_size': 128,
  'extra_num_layers': 0,
  'image_encoder': { 'network': 'ResnetEncoder',
                     'network_kwargs': { 'freeze': False,
                                         'language_fusion': 'film',
                                         'no_stride': False,
                                         'pretrained': False,
                                         'remove_layer_num': 4}},
  'language_encoder': { 'network': 'MLPEncoder',
                        'network_kwargs': { 'hidden_size': 128,
                                            'input_size': 768,
 

## 2. 编写你自己的策略网络结构

In [ ]:
# 这个代码单元演示：如何在 LIBERO 里“自己写一个 policy 网络结构”（以 Transformer policy 为例）
# 你可以把它理解成：把观测（图像/语言/关节等）编码成 token，然后用时序 Transformer 做决策。

# TensorUtils：robomimic 提供的张量维度工具（例如把多个维度拼到一起）
import robomimic.utils.tensor_utils as TensorUtils
# PyTorch 主包
import torch
# PyTorch 的神经网络模块（Linear / ReLU / Module / ModuleList 等）
import torch.nn as nn

# einops：更直观的张量重排工具（rearrange / repeat）；本示例里可能不一定都用到
from einops import rearrange, repeat
# LIBERO 自带的 RGB 编码模块（这里用 * 方便示例；真实项目建议显式导入用到的类）
from libero.lifelong.models.modules.rgb_modules import *
# LIBERO 自带的语言编码模块
from libero.lifelong.models.modules.language_modules import *
# BasePolicy：LIBERO policy 的基类（包含 preprocess_input 等通用逻辑）
from libero.lifelong.models.base_policy import BasePolicy
# policy head：把隐状态映射到动作分布/动作向量的输出头
from libero.lifelong.models.policy_head import *
# Transformer 相关模块（Decoder、位置编码等）
from libero.lifelong.models.modules.transformer_modules import *

###############################################################################
#
# A model handling extra input modalities besides images at time t.
# （除了图像之外，还可能有：关节角、夹爪开合、末端执行器位置等额外输入）
#
###############################################################################

class ExtraModalityTokens(nn.Module):
    # 这个模块的作用：把“非图像”的低维观测（proprioception）映射成 token（embedding），
    # 这样就能和图像 token、语言 token 放到同一个 Transformer 里处理。
    def __init__(
        self,
        use_joint=False,      # 是否使用关节角（默认 False）
        use_gripper=False,    # 是否使用夹爪状态（默认 False）
        use_ee=False,         # 是否使用末端位置（默认 False）
        extra_num_layers=0,   # 额外模态的 MLP 层数（0 表示只用一层 Linear）
        extra_hidden_size=64, # MLP 隐藏层维度
        extra_embedding_size=32, # 输出 token 的 embedding 维度
    ):
        """
        This is a class that maps all extra modality inputs into tokens of the same size
        """
        super().__init__()  # 调用 nn.Module 的初始化
        self.use_joint = use_joint  # 记录是否启用 joint_states
        self.use_gripper = use_gripper  # 记录是否启用 gripper_states
        self.use_ee = use_ee  # 记录是否启用 ee_states
        self.extra_embedding_size = extra_embedding_size  # 记录输出 embedding 大小

        joint_states_dim = 7    # 机器人关节角维度（Franka 常见是 7）
        gripper_states_dim = 2  # 夹爪状态维度（例如左右指的开合）
        ee_dim = 3              # 末端执行器位置维度（x, y, z）

        # 计算“启用了多少种额外模态”，后面会用来决定 token 数量
        self.num_extra = int(use_joint) + int(use_gripper) + int(use_ee)

        # 计算所有启用模态拼起来的原始低维特征总维度
        extra_low_level_feature_dim = (
            int(use_joint) * joint_states_dim
            + int(use_gripper) * gripper_states_dim
            + int(use_ee) * ee_dim
        )

        # 如果没有启用任何模态，这个模块就没有意义；直接报错提示
        assert extra_low_level_feature_dim > 0, "[error] no extra information"

        # extra_encoders：用一个 dict 保存每种模态对应的 encoder（MLP）
        self.extra_encoders = {}

        # 生成“把某个 proprio 模态 -> embedding”的 MLP 的小函数
        def generate_proprio_mlp_fn(modality_name, extra_low_level_feature_dim):
            # 再检查一次：维度必须 > 0
            assert extra_low_level_feature_dim > 0  # we indeed have extra information
            # 如果 extra_num_layers > 0：构建多层 MLP
            if extra_num_layers > 0:
                # 第一层：原始维度 -> hidden
                layers = [nn.Linear(extra_low_level_feature_dim, extra_hidden_size)]
                # 中间层：hidden -> hidden，并接 ReLU
                for i in range(1, extra_num_layers):
                    layers += [
                        nn.Linear(extra_hidden_size, extra_hidden_size),
                        nn.ReLU(inplace=True),
                    ]
                # 最后一层：hidden -> embedding
                layers += [nn.Linear(extra_hidden_size, extra_embedding_size)]
            else:
                # 如果 extra_num_layers == 0：只用一层 Linear 直接映射到 embedding
                layers = [nn.Linear(extra_low_level_feature_dim, extra_embedding_size)]

            # 把 layers 变成 nn.Sequential，方便 forward 直接调用
            self.proprio_mlp = nn.Sequential(*layers)
            # 存到 dict 里，方便按 modality_name 取出
            self.extra_encoders[modality_name] = {"encoder": self.proprio_mlp}

        # 遍历三种可能的 proprio 模态：joint / gripper / ee
        for (proprio_dim, use_modality, modality_name) in [
            (joint_states_dim, self.use_joint, "joint_states"),
            (gripper_states_dim, self.use_gripper, "gripper_states"),
            (ee_dim, self.use_ee, "ee_states"),
        ]:

            # 如果用户在 cfg 里启用了该模态，就创建对应 MLP encoder
            if use_modality:
                generate_proprio_mlp_fn(modality_name, proprio_dim)

        # 把所有 encoder 收集到 ModuleList：这样它们的参数才能被 PyTorch 正确注册/保存/加载
        self.encoders = nn.ModuleList(
            [x["encoder"] for x in self.extra_encoders.values()]
        )

    def forward(self, obs_dict):
        """
        obs_dict: {
            (optional) joint_stats: (B, T, 7),
            (optional) gripper_states: (B, T, 2),
            (optional) ee: (B, T, 3)
        }
        map above to a latent vector of shape (B, T, H)
        """
        # tensor_list：把每个启用模态编码后的 token 暂存起来
        tensor_list = []

        # 按固定顺序检查每种模态是否启用
        for (use_modality, modality_name) in [
            (self.use_joint, "joint_states"),
            (self.use_gripper, "gripper_states"),
            (self.use_ee, "ee_states"),
        ]:

            if use_modality:
                # obs_dict[modality_name] 的形状通常是 (B, T, dim)
                # encoder 输出形状会变成 (B, T, E)
                tensor_list.append(
                    self.extra_encoders[modality_name]["encoder"](
                        obs_dict[modality_name]
                    )
                )

        # 把不同模态沿着“模态维度”堆叠起来：得到 (B, T, num_extra, E)
        x = torch.stack(tensor_list, dim=-2)
        # 返回 token 序列（供后面的 Transformer 使用）
        return x

###############################################################################
#
# A Transformer policy
# （主角：把语言 + 额外模态 + 图像编码成 token，再做时序 Transformer）
#
###############################################################################

class MyTransformerPolicy(BasePolicy):
    """
    Input: (o_{t-H}, ... , o_t)   # 输入：一段历史观测序列
    Output: a_t or distribution of a_t  # 输出：当前动作或动作分布
    """

    def __init__(self, cfg, shape_meta):
        # BasePolicy 会做一些通用初始化（例如保存 cfg、shape_meta 等）
        super().__init__(cfg, shape_meta)
        # 取一个更短的名字，方便后面写配置字段
        policy_cfg = cfg.policy

        ### 1. encode image
# 第一步：把每个摄像头图像（rgb/depth）编码成 embedding token
        embed_size = policy_cfg.embed_size  # 统一的 token embedding 维度（E）
        transformer_input_sizes = []  # 这个列表在本示例里未使用，保留作扩展
        self.image_encoders = {}  # 用 dict 保存每个图像模态对应的 encoder
        for name in shape_meta["all_shapes"].keys():  # 遍历所有观测名字（例如 agentview_rgb）
            if "rgb" in name or "depth" in name:  # 只处理图像相关模态
                kwargs = policy_cfg.image_encoder.network_kwargs  # 取出图像 encoder 的网络参数模板
                kwargs.input_shape = shape_meta["all_shapes"][name]  # 设定输入图像形状 (C,H,W)
                kwargs.output_size = embed_size  # 设定输出 embedding 维度 E
                kwargs.language_dim = (
                    policy_cfg.language_encoder.network_kwargs.input_size  # 语言 embedding 输入维度（用于图文融合时）
                )
                # 保存 encoder：eval() 根据配置里的字符串类名实例化网络（示例里图省事）
                self.image_encoders[name] = {
                    "input_shape": shape_meta["all_shapes"][name],
                    "encoder": eval(policy_cfg.image_encoder.network)(**kwargs),
                }

        # 同样把所有图像 encoder 注册到 ModuleList，保证参数可训练/可保存
        self.encoders = nn.ModuleList(
            [x["encoder"] for x in self.image_encoders.values()]
        )

        ### 2. encode language
# 第二步：语言编码器，把任务描述文本 -> embedding（E）
        policy_cfg.language_encoder.network_kwargs.output_size = embed_size  # 让语言编码输出也变成 E
        self.language_encoder = eval(policy_cfg.language_encoder.network)(
            **policy_cfg.language_encoder.network_kwargs  # 把配置里的参数展开传入构造函数
        )

        ### 3. encode extra information (e.g. gripper, joint_state)
# 第三步：把关节/夹爪/末端等低维观测 -> token（E）
        self.extra_encoder = ExtraModalityTokens(
            use_joint=cfg.data.use_joint,            # 是否用 joint_states
            use_gripper=cfg.data.use_gripper,        # 是否用 gripper_states
            use_ee=cfg.data.use_ee,                  # 是否用 ee_states
            extra_num_layers=policy_cfg.extra_num_layers,      # 额外模态 MLP 层数
            extra_hidden_size=policy_cfg.extra_hidden_size,    # MLP 隐藏维度
            extra_embedding_size=embed_size,                   # 输出 embedding 维度 E（与其它 token 对齐）
        )

        ### 4. define temporal transformer
# 第四步：定义“时间维度”的 Transformer（对序列 token 做建模）
        policy_cfg.temporal_position_encoding.network_kwargs.input_size = embed_size  # 位置编码输入维度是 E
        self.temporal_position_encoding_fn = eval(
            policy_cfg.temporal_position_encoding.network  # 从字符串名构造位置编码模块
        )(**policy_cfg.temporal_position_encoding.network_kwargs)

        # TransformerDecoder：对 (B, seq, E) 做自注意力，输出同形状隐藏表示
        self.temporal_transformer = TransformerDecoder(
            input_size=embed_size,
            num_layers=policy_cfg.transformer_num_layers,          # Transformer 层数
            num_heads=policy_cfg.transformer_num_heads,            # 注意力头数
            head_output_size=policy_cfg.transformer_head_output_size, # 每个 head 的输出维度
            mlp_hidden_size=policy_cfg.transformer_mlp_hidden_size, # FFN 隐藏层维度
            dropout=policy_cfg.transformer_dropout,                # dropout 概率
        )

        # policy head：把 Transformer 输出的 embedding -> 动作分布/动作向量
        policy_head_kwargs = policy_cfg.policy_head.network_kwargs  # 输出头的网络参数
        policy_head_kwargs.input_size = embed_size  # 输入维度 E
        policy_head_kwargs.output_size = shape_meta["ac_dim"]  # 输出维度 = 动作维度（例如 7）

        # eval 根据配置构建输出头；loss_kwargs 可能包含分布类型/损失相关参数
        self.policy_head = eval(policy_cfg.policy_head.network)(
            **policy_cfg.policy_head.loss_kwargs,
            **policy_cfg.policy_head.network_kwargs
        )

        # latent_queue：在线推理时用来缓存过去若干步的 token（实现滑动窗口）
        self.latent_queue = []
        # 最大缓存长度（也就是 Transformer 看到的最大序列长度）
        self.max_seq_len = policy_cfg.transformer_max_seq_len

    def temporal_encode(self, x):
        # x 形状：(B, T, num_modality, E) —— 每个时间步有多个模态 token
        # 生成时序位置编码（告诉 Transformer 当前 token 是第几步）
        pos_emb = self.temporal_position_encoding_fn(x)
        # 把位置编码加到 token 上；unsqueeze(1) 用来对齐维度，便于广播相加
        x = x + pos_emb.unsqueeze(1)  # (B, T, num_modality, E)
        # 记录原始形状，后面 Transformer 跑完要 reshape 回来
        sh = x.shape
        # 让 Transformer 根据当前序列长度计算 attention mask（例如 causal mask）
        self.temporal_transformer.compute_mask(x.shape)

        # 把 (T, num_modality) 两个维度拼成一个序列维度： (B, T*num_modality, E)
        x = TensorUtils.join_dimensions(x, 1, 2)  # (B, T*num_modality, E)
        # 送入 Transformer 做时序建模
        x = self.temporal_transformer(x)
        # 再 reshape 回 (B, T, num_modality, E)
        x = x.reshape(*sh)
        # 这里只取第 0 个模态 token 作为“聚合后的时序表示”（约定：第一个 token 是语言/动作 token）
        return x[:, :, 0]  # (B, T, E)

    def spatial_encode(self, data):
        # 这个函数负责：把原始输入 data -> 多模态 token（不做时间 Transformer）
        # data 通常包含：data["obs"]（观测字典）、data["task_emb"]（语言/任务 embedding）等

        # 1. encode extra：把 low-dim proprio 观测编码成 token
        extra = self.extra_encoder(data["obs"])  # (B, T, num_extra, E)

        # 2. encode language：把语言 embedding 当成一个 token（有点像 [CLS] 或 action token）
        B, T = extra.shape[:2]  # 取 batch 大小 B 和序列长度 T
        text_encoded = self.language_encoder(data)  # (B, E) 语言编码结果
        # view + expand：把 (B,E) 扩成 (B,T,1,E)，让每个时间步都带同一个语言 token
        text_encoded = text_encoded.view(B, 1, 1, -1).expand(
            -1, T, -1, -1
        )  # (B, T, 1, E)
        # 先把语言 token 和 extra token 放到列表里，后面再加图像 token
        encoded = [text_encoded, extra]

        # 3. encode image：遍历每个图像模态，把每帧图像编码成 token
        for img_name in self.image_encoders.keys():
            # 取出该模态的图像序列：形状一般是 (B, T, C, H, W)
            x = data["obs"][img_name]
            # 拆出各维度，便于 reshape
            B, T, C, H, W = x.shape
            # 把 (B,T,...) 展平成 (B*T,...) 批处理编码（更高效）
            img_encoded = self.image_encoders[img_name]["encoder"](
                x.reshape(B * T, C, H, W),  # 图像输入：(B*T, C, H, W)
                # langs：把 task_emb 也扩展到每一帧，并展平到 (B*T, lang_dim)
                langs=data["task_emb"]
                .reshape(B, 1, -1)
                .repeat(1, T, 1)
                .reshape(B * T, -1),
            ).view(B, T, 1, -1)  # 编码后再 reshape 回 (B, T, 1, E)
            # 把图像 token 追加到 encoded 列表
            encoded.append(img_encoded)
        # 把所有 token 在“模态维度”拼接：得到 (B, T, num_modalities, E)
        encoded = torch.cat(encoded, -2)  # (B, T, num_modalities, E)
        # 返回多模态 token
        return encoded

    def forward(self, data):
        # forward：训练时常用；输入 data，输出动作分布 dist
        x = self.spatial_encode(data)   # 先做空间/模态编码 -> token
        x = self.temporal_encode(x)     # 再做时间编码 -> (B,T,E)
        dist = self.policy_head(x)      # 输出头：把 embedding -> 动作分布/动作
        return dist

    def get_action(self, data):
        # get_action：推理时常用；输出一个可执行的动作（numpy）
        self.eval()  # 切换到 eval 模式（关闭 dropout 等）
        with torch.no_grad():  # 推理不需要梯度，省显存/更快
            # preprocess_input：BasePolicy 提供的输入预处理（归一化、转 device、整理字典结构等）
            data = self.preprocess_input(data, train_mode=False)
            # 先做空间编码，得到当前时间步的 token（注意：这里是单步输入，也会带一个 T 维）
            x = self.spatial_encode(data)
            # 把当前 token 加入队列，用来形成一个滑动窗口序列
            self.latent_queue.append(x)
            # 如果超过最大序列长度，就丢掉最早的 token
            if len(self.latent_queue) > self.max_seq_len:
                self.latent_queue.pop(0)
            # 沿时间维拼起来：得到 (B, T_window, num_modalities, E)
            x = torch.cat(self.latent_queue, dim=1)  # (B, T, H_all)
            # 时间 Transformer 编码
            x = self.temporal_encode(x)
            # 只取最后一步的 embedding 来预测当前动作
            dist = self.policy_head(x[:, -1])
        # 从分布里 sample 一个动作，并搬到 CPU（便于转 numpy）
        action = dist.sample().detach().cpu()
        # 展平动作维度，转成 numpy（LIBERO 环境 step 通常要 numpy）
        return action.view(action.shape[0], -1).numpy()

    def reset(self):
        # 每个 episode/每个任务开始时最好 reset：清空历史缓存
        self.latent_queue = []

## 3. 编写你自己的终身学习算法

In [ ]:
# 这个代码单元演示：如何在 LIBERO 的 lifelong 学习框架里自定义一个算法（Algo）
# 核心思想：所有算法都继承 Sequential，然后实现/覆写几个关键接口。

# Sequential：LIBERO 提供的“按任务顺序逐个学习”的算法基类
from libero.lifelong.algos.base import Sequential

### All lifelong learning algorithm should inherit the Sequential algorithm super class
# 上面这行英文提示的意思是：所有 lifelong 学习算法都应该继承 Sequential

class MyLifelongAlgo(Sequential):
    """
    The experience replay policy.
    """
    # 这里 __init__ 负责：创建 policy、准备算法自身要保存的状态（如 replay buffer / datasets 等）
    def __init__(self,
                 n_tasks,          # 总任务数（benchmark.n_tasks）
                 cfg,              # Hydra 配置（里面包含数据、训练、模型等信息）
                 **policy_kwargs): # 额外传给 policy 的参数（可选）
        # 先调用父类初始化：Sequential 会设置通用字段、优化器等（具体取决于实现）
        super().__init__(n_tasks=n_tasks, cfg=cfg, **policy_kwargs)
        # 用一个列表保存“已经学过的任务的数据集”（这里相当于最简单的经验回放示例）
        # define the learning policy
        self.datasets = []
        # 根据 cfg.policy.policy_type 里写的类名，动态构造 policy（本 notebook 上一节定义了 MyTransformerPolicy）
        # 注意：cfg.shape_meta 需要先在训练脚本里设好（来自 get_dataset 返回的 shape_meta）
        self.policy = eval(cfg.policy.policy_type)(cfg, cfg.shape_meta)

    def start_task(self, task):
        # start_task：开始学习一个新任务时会被调用
        # 你可以在这里做：重置优化器、初始化任务特定的参数、清空临时缓存等
        super().start_task(task)

    def end_task(self, dataset, task_id, benchmark):
        # end_task：完成一个任务训练后会被调用
        # 你可以在这里做：把当前任务的数据加入 replay buffer、做模型冻结、做正则项统计等
        self.datasets.append(dataset)

    def observe(self, data):
        # observe：算法每次看到一个 batch 数据时怎么计算 loss
        # 在最基础的 BC（behavior cloning）里，父类 Sequential.observe(data) 通常会：
        # 1) 把 data 喂给 policy 得到动作分布/预测动作；2) 计算模仿学习损失；3) 返回 loss
        loss = super().observe(data)
        # 也可以在这里加上额外正则项（EWC / L2 / distillation 等）
        return loss

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/yifengz/workspace/robosuite-master/robosuite/scripts/setup_macros.py (__init__.py:9)


# 4. 编写你的训练脚本

In [ ]:
# 这个代码单元演示：如何真正跑起来训练（按任务顺序逐个学习）

# 指定 policy 类型：告诉系统用我们刚刚定义的 MyTransformerPolicy
cfg.policy.policy_type = "MyTransformerPolicy"
# 指定 lifelong 算法类型：告诉系统用我们刚刚定义的 MyLifelongAlgo
cfg.lifelong.algo = "MyLifelongAlgo"

# 创建实验输出目录（日志、模型权重、结果文件等会放在 cfg.experiment_dir）
create_experiment_dir(cfg)
# 把数据集返回的 shape_meta 写回 cfg：policy 构造时需要知道动作维度/观测形状
cfg.shape_meta = shape_meta

# numpy：用于创建/操作结果矩阵
import numpy as np
# trange：tqdm 提供的带进度条的 range
from tqdm import trange
# 评估工具：evaluate_loss 评估 BC loss；evaluate_success 评估环境 rollout 成功率
from libero.lifelong.metric import evaluate_loss, evaluate_success

# 打印实验目录，方便你去磁盘里找输出文件
print("experiment directory is: ", cfg.experiment_dir)
# 构造算法对象，并用 safe_device 放到 cfg.device（例如 cuda:0 或 cpu）
algo = safe_device(MyLifelongAlgo(n_tasks, cfg), cfg.device)

# result_summary：用于汇总所有任务学习过程中的指标
result_summary = {
    'L_conf_mat': np.zeros((n_tasks, n_tasks)),   # loss confusion matrix：第 i 行第 j 列 = 训练到任务 i 后，在任务 j 上的 loss
    'S_conf_mat': np.zeros((n_tasks, n_tasks)),   # success confusion matrix：第 i 行第 j 列 = 训练到任务 i 后，在任务 j 上的成功率
    'L_fwd'     : np.zeros((n_tasks,)),           # forward AUC（loss）：学习当前任务时 loss 曲线的面积（越低越好，代表学得快）
    'S_fwd'     : np.zeros((n_tasks,)),           # forward AUC（success）：学习当前任务时成功率曲线的面积（越高越好）
}

# task_group_size：LIBERO 里有些 benchmark 会把任务按 group 组织；评估成功率时会用到索引映射
gsz = cfg.data.task_group_size

# 下面两段只是打印提示：如果训练轮数/评估回合数偏小，会提醒你这是为了示例简化
if (cfg.train.n_epochs < 50):
    print("NOTE: the number of epochs used in this example is intentionally reduced to 30 for simplicity.")
if (cfg.eval.n_eval < 20):
    print("NOTE: the number of evaluation episodes used in this example is intentionally reduced to 5 for simplicity.")

# 按任务顺序逐个训练（lifelong/continual setting）
for i in trange(n_tasks):
    # 进入训练模式（启用 dropout 等）
    algo.train()
    # learn_one_task：Sequential 系列算法的关键函数：在第 i 个任务的数据集上训练，并返回该任务的 forward 指标
    s_fwd, l_fwd = algo.learn_one_task(datasets[i], i, benchmark, result_summary)
    # s_fwd：当前任务的成功率 AUC（通常按若干评估间隔采样）
    # s_fwd is success rate AUC, when the agent learns the {0, e, 2e, ...} epochs
    # l_fwd：当前任务的 BC loss AUC
    # l_fwd is BC loss AUC, similar to s_fwd
    result_summary["S_fwd"][i] = s_fwd
    result_summary["L_fwd"][i] = l_fwd

    # 如果配置里打开了评估，就在学完任务 i 后评估“到目前为止的所有任务”
    if cfg.eval.eval:
        # 进入评估模式（关闭 dropout）
        algo.eval()
        # we only evaluate on the past tasks: 0 .. i
        # evaluate_loss：返回长度 (i+1) 的数组，对应任务 0..i 的 loss
        L = evaluate_loss(cfg, algo, benchmark, datasets[:i+1]) # (i+1,)
        # evaluate_success：返回长度 (i+1) 的数组，对应任务 0..i 的成功率
        # 这里用 list(range((i+1)*gsz)) 把“任务组索引”展开成具体任务索引（与 benchmark 定义有关）
        S = evaluate_success(cfg, algo, benchmark, list(range((i+1)*gsz))) # (i+1,)
        # 把当前评估结果写入混淆矩阵：第 i 行表示“学到第 i 个任务之后”的整体表现
        result_summary["L_conf_mat"][i][:i+1] = L
        result_summary["S_conf_mat"][i][:i+1] = S

        # 把结果保存到实验目录，方便后面可视化/计算 FWT/BWT/AUC 等指标
        torch.save(result_summary, os.path.join(cfg.experiment_dir, f'result.pt'))

experiment directory is:  ./experiments/libero_object/MyLifelongAlgo/MyTransformerPolicy_seed10000/run_006


/home/yifengz/.pyenv/versions/3.8.5/envs/new-continual-learning/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
/home/yifengz/.pyenv/versions/3.8.5/envs/new-continual-learning/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/yifengz/.pyenv/versions/3.8.5/envs/new-continual-learning/lib/python3.8/site-packages/torch/functional.py:478: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at  ../aten/src/ATen/native/TensorShape.cpp:2894.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


NOTE: the number of epochs used in this example is intentionally reduced to 30 for simplicity.
NOTE: the number of evaluation episodes used in this example is intentionally reduced to 5 for simplicity.


  0%|          | 0/10 [00:00<?, ?it/s]

[info] Epoch:   0 | train loss:  5.45 | time: 0.64
[info] evaluate task 0 takes 109.5 seconds
[info] Epoch:   0 | succ: 0.00 ± 0.00 | best succ: 0.0 | succ. AoC 0.00 | time: 1.82
[info] Epoch:   1 | train loss: -6.00 | time: 1.32
[info] Epoch:   2 | train loss: -12.35 | time: 1.31
[info] Epoch:   3 | train loss: -14.18 | time: 1.30
[info] Epoch:   4 | train loss: -14.78 | time: 1.30
[info] Epoch:   5 | train loss: -15.23 | time: 1.28
[info] evaluate task 0 takes 105.7 seconds
[info] Epoch:   5 | succ: 0.00 ± 0.00 | best succ: 0.0 | succ. AoC 0.00 | time: 1.76
[info] Epoch:   6 | train loss: -15.64 | time: 1.22
[info] Epoch:   7 | train loss: -16.03 | time: 1.22
[info] Epoch:   8 | train loss: -16.58 | time: 1.22
[info] Epoch:   9 | train loss: -16.81 | time: 1.22
[info] Epoch:  10 | train loss: -17.12 | time: 1.22
[info] evaluate task 0 takes 68.6 seconds
[info] Epoch:  10 | succ: 0.60 ± 0.43 | best succ: 0.6 | succ. AoC 0.20 | time: 1.14
[info] Epoch:  11 | train loss: -17.29 | time: 

 10%|█         | 1/10 [41:46<6:15:58, 2506.46s/it]

... (15 outputs left out)

 90%|█████████ | 9/10 [7:08:19<52:24, 3144.50s/it]  

[info] evaluate task 8 takes 55.9 seconds
[info] Epoch:   0 | train loss: 49.85 | time: 0.54
[info] evaluate task 9 takes 105.7 seconds
[info] Epoch:   0 | succ: 0.00 ± 0.00 | best succ: 0.0 | succ. AoC 0.00 | time: 1.76
[info] Epoch:   1 | train loss: -17.30 | time: 1.08
[info] Epoch:   2 | train loss: -18.84 | time: 1.08
[info] Epoch:   3 | train loss: -19.23 | time: 1.09
[info] Epoch:   4 | train loss: -19.55 | time: 1.09
[info] Epoch:   5 | train loss: -19.99 | time: 1.10
[info] evaluate task 9 takes 34.9 seconds
[info] Epoch:   5 | succ: 1.00 ± 0.00 | best succ: 1.0 | succ. AoC 0.50 | time: 0.58
[info] Epoch:   6 | train loss: -20.42 | time: 1.10
[info] Epoch:   7 | train loss: -20.64 | time: 1.10
[info] Epoch:   8 | train loss: -21.00 | time: 1.09
[info] Epoch:   9 | train loss: -21.22 | time: 1.09
[info] Epoch:  10 | train loss: -21.49 | time: 1.09
[info] evaluate task 9 takes 65.4 seconds
[info] Epoch:  10 | succ: 0.60 ± 0.43 | best succ: 1.0 | succ. AoC 0.67 | time: 1.09
[info

100%|██████████| 10/10 [8:00:57<00:00, 2885.72s/it]

[info] evaluate task 9 takes 33.5 seconds


# 5. 可视化结果

### 5.1 加载结果

In [ ]:
# 从实验目录读取刚才保存的 result.pt（里面是一个 dict，包含混淆矩阵和 forward 指标）
result_summary = torch.load(os.path.join(cfg.experiment_dir, f'result.pt'))
# 打印成功率混淆矩阵：第 i 行表示“学到第 i 个任务后”，在各任务上的成功率
print(result_summary["S_conf_mat"])
# 打印每个任务的 forward AUC（成功率）
print(result_summary["S_fwd"])

[[0.6 0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.8 0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.8 0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.2 0.6 0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.8 0.  0.  0.  0.  0. ]
 [0.  0.  0.  1.  0.  0.8 0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.2 0.  0.4 0.  0.  0. ]
 [0.4 0.  0.  0.  0.8 0.  0.2 0.6 0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.4 0.  0.8 0. ]
 [0.  0.  0.8 0.2 0.  0.  0.  0.  0.  1. ]]
[0.5        0.63333333 0.66666667 0.33333333 0.56666667 0.76666667
 0.33333333 0.66666667 0.73333333 0.83333333]


### 5.2 计算实验的 FWT、BWT 和 AUC

In [ ]:
# 这个代码单元：定义一些“名字映射表”和“指标计算函数”，用于从实验结果里算 FWT / BWT / AUC。

# torch：读取保存的结果（torch.load）以及做张量运算
import torch
# numpy：主要用来做数组统计/均值
import numpy as np
# Path：路径工具（这里导入了但本单元未实际使用）
from pathlib import Path

# benchmark_map：把配置里的 benchmark 名称映射到更好看的展示名
benchmark_map = {
    "libero_10"     : "LIBERO_10",
    "libero_90"     : "LIBERO_90",
    "libero_spatial": "LIBERO_SPATIAL",
    "libero_object" : "LIBERO_OBJECT",
    "libero_goal"   : "LIBERO_GOAL",
}

# algo_map：把算法内部名映射到论文/展示中常用的名字
algo_map = {
    "base"     : "Sequential",
    "er"       : "ER",
    "ewc"      : "EWC",
    "packnet"  : "PackNet",
    "multitask": "Multitask",
    "custom_algo"   : "MyLifelongAlgo",
}

# policy_map：把 policy 配置名映射到类名（或展示名）
policy_map = {
    "bc_rnn_policy"        : "BCRNNPolicy",
    "bc_transformer_policy": "BCTransformerPolicy",
    "bc_vilt_policy"       : "BCViLTPolicy",
    "custom_policy"        : "MyTransformerPolicy",
}

# seeds：实验随机种子列表（这里示例只放一个种子 10000）
seeds = [10000]
# N_SEEDS：种子数量
N_SEEDS = len(seeds)
# N_TASKS：假设任务数为 10（这里写死是为了示例；更通用做法是用 benchmark.n_tasks）
N_TASKS = 10

def get_auc(experiment_dir, bench, algo, policy):
    # N_EP：评估点的个数 = 总 epoch 数 // 每隔多少 epoch 评估一次 + 1（包含 epoch=0 的初始点）
    N_EP = cfg.train.n_epochs // cfg.eval.eval_every + 1
    # fwds：保存每个任务、每个评估点、每个 seed 的成功率曲线
    fwds = np.zeros((N_TASKS, N_EP, N_SEEDS))

    # 遍历每个任务，读取该任务的 AUC 日志文件
    for task in range(N_TASKS):
        counter = 0  # 这个变量在示例里未使用（保留项）
        # 遍历每个随机种子
        for k, seed in enumerate(seeds):
            # name：日志文件名（示例格式：experiments/task{task}_auc.log）
            name = f"{experiment_dir}/task{task}_auc.log"
            try:
                # torch.load 读取日志，假设里面有 key "success"，长度为 n_epochs 或评估点数
                succ = torch.load(name)["success"] # (n_epochs)
                # idx：成功率最高的位置（argmax）
                idx = succ.argmax()
                # 把最高点之后的成功率都“拉平”为最高值：避免后面下降影响面积（这是一个常见的平滑/截断技巧）
                succ[idx:] = succ[idx]
                # 写入 fwds：第 task 个任务、第 k 个 seed 的成功率曲线
                fwds[task, :, k] = succ
            except:
                # 如果文件不存在或格式不对，就提示并跳过
                print("Some errors when loading results")
                continue
    # 返回所有任务的 forward success 曲线（用于算 FWT/AUC 等）
    return fwds

def compute_metric(res):
    # res = (mat, fwts)
    # mat：混淆矩阵（success 或 loss），形状通常是 (T_trained, T_eval, seed) 或类似结构
    # fwts：每个任务自己的 forward 曲线（success），形状 (num_tasks, num_eval_points, num_seeds)
    mat, fwts  = res # fwds: (num_tasks, num_save_intervals, num_seeds)
    # 从 mat 的 shape 推出：任务数、种子数（这里假设 mat 的后两维分别是任务、seed）
    num_tasks, num_seeds = mat.shape[1:]
    # ret：最终要返回的指标字典
    ret = {}

    # compute fwt（Forward Transfer）：衡量“学新任务是否帮助未来任务/泛化”
    # 这里用 fwts 的均值作为一个简化版的 forward-transfer 指标
    fwt = fwts.mean(axis=(0,1))
    ret["fwt"] = fwt
    # compute bwt（Backward Transfer）：衡量“学新任务对旧任务的影响”（遗忘/正迁移）
    bwts = []
    aucs = []
    # 对每个 seed 分别计算，再汇总成数组
    for seed in range(num_seeds):
        bwt = 0.0
        auc = 0.0
        # 遍历每个任务 k，当作“旧任务”
        for k in range(num_tasks):
            bwt_k = 0.0
            auc_k = 0.0
            # tau 表示“后来学到的任务”，tau > k
            for tau in range(k+1, num_tasks):
                # mat[k,k,seed]：学完任务 k 之后在任务 k 上的表现
                # mat[tau,k,seed]：学完任务 tau 之后在任务 k 上的表现
                # 二者差值表示遗忘/后向迁移（越大通常表示遗忘越严重）
                bwt_k += mat[k,k,seed] - mat[tau,k,seed]
                # 累加用于 auc_k 的项（这里把后续阶段在旧任务上的表现累计起来）
                auc_k += mat[tau,k,seed]
            # 归一化：对 k 之后还有多少个任务进行平均
            if k + 1 < num_tasks:
                bwt_k /= (num_tasks - k - 1)
            # auc_k：把混淆矩阵累计值与 forward 曲线均值结合，构成一个综合 AUC 指标
            auc_k = (auc_k + fwts[k,:,seed].mean()) / (num_tasks - k)

            # 把当前任务 k 的贡献累加到总体 bwt/auc
            bwt += bwt_k
            auc += auc_k
        # 对任务数取平均
        bwts.append(bwt / num_tasks)
        aucs.append(auc / num_tasks)
    # 转成 numpy 数组，方便后续统计均值/方差
    bwts = np.array(bwts)
    aucs = np.array(aucs)
    # 写入返回 dict
    ret["bwt"] = bwts
    ret["auc"] = aucs
    return ret

In [ ]:
# 下面这些变量相当于“告诉指标脚本：你要看哪个实验/哪个 benchmark/哪个算法/哪个 policy”
experiment_dir = "experiments"          # 默认实验输出根目录（示例变量）
benchmark_name = "libero_object"       # benchmark 名称
algo_name = "custom_algo"              # 算法名（对应上面 algo_map 里的 key）
policy_name = "custom_policy"          # policy 名（对应上面 policy_map 里的 key）

# 读取 forward success 曲线（fwds）：形状大致是 (N_TASKS, N_EP, N_SEEDS)
fwds = get_auc(cfg.experiment_dir, benchmark_name, algo_name, policy_name)

# conf_mat：从 result_summary 里取成功率混淆矩阵 S_conf_mat
# [..., np.newaxis]：在最后加一个维度，变成 (n_tasks, n_tasks, 1) 以匹配 compute_metric 里对 seed 维度的假设
conf_mat = result_summary["S_conf_mat"][..., np.newaxis]

# 计算指标：返回 dict，包含 fwt / bwt / auc
metric = compute_metric((conf_mat, fwds))
# 打印指标结果
print(metric)


{'fwt': array([0.60333333]), 'bwt': array([0.54126984]), 'auc': array([0.26577513])}


### 5.3 可视化策略执行轨迹（rollout）

这是一个如何使用训练好的模型进行推理的示例。我们将以在第一个任务上训练得到的策略作为示例。更具体的例子，请参见 `lifelong/lifelong/metric.py` 文件中的 `evaluate_one_task_success`。

In [ ]:
# 这个代码单元演示：用训练好的 policy 在环境里 rollout，并把过程录成视频在 notebook 里展示

# HTML：用于在 notebook 里直接显示 <video> 标签
from IPython.display import HTML
# b64encode：把二进制 mp4 转成 base64 字符串嵌入 HTML
from base64 import b64encode
# imageio：写视频文件（把一帧帧图像编码成 mp4）
import imageio

# OffScreenRenderEnv：离屏渲染环境（不需要窗口/显示器）
# DummyVectorEnv：把单个环境包装成“向量化环境”接口（即使 env_num=1 也用统一接口）
from libero.libero.envs import OffScreenRenderEnv, DummyVectorEnv
# raw_obs_to_tensor_obs：把环境返回的原始 obs（numpy/dict）整理成 policy 需要的张量输入
from libero.lifelong.metric import raw_obs_to_tensor_obs

# You can turn on subprocess
# env_num：并行环境数量（这里设 1，最简单）
env_num = 1
# action_dim：动作维度（Franka 常见为 7；要和 shape_meta["ac_dim"] 一致）
action_dim = 7

# If it's packnet, the weights need to be processed first
# task_id：选择要测试的任务编号（这里挑第 9 个任务）
task_id = 9
# 从 benchmark 拿到任务对象（包含 problem_folder、bddl_file 等）
task = benchmark.get_task(task_id)
# 拿到该任务对应的语言/任务 embedding（policy 推理需要）
task_emb = benchmark.get_task_emb(task_id)

# PackNet 这类算法：不同任务可能需要切换“评估用子网络/掩码”；所以要获取专用的 eval algo
if cfg.lifelong.algo == "PackNet":
    algo = algo.get_eval_algo(task_id)

# 切到 eval 模式：推理时关闭 dropout 等
algo.eval()
# env_args：构造环境所需的参数字典
env_args = {
    # bddl_file_name：该任务的 BDDL 文件路径（描述任务目标/对象/约束）
    "bddl_file_name": os.path.join(
        cfg.bddl_folder, task.problem_folder, task.bddl_file
    ),
    # camera_heights / camera_widths：渲染图像的高宽（与训练数据一致很重要）
    "camera_heights": cfg.data.img_h,
    "camera_widths": cfg.data.img_w,
}

# 创建向量环境：这里 env_num=1，所以列表里只有一个 lambda 创建的环境实例
env = DummyVectorEnv(
            [lambda: OffScreenRenderEnv(**env_args) for _ in range(env_num)]
)

# init_states_path：该任务的初始状态文件路径（用于复现实验/固定起点评估）
init_states_path = os.path.join(
    cfg.init_states_folder, task.problem_folder, task.init_states_file
)
# 读取初始化状态集合（通常是一个 tensor/数组，包含多个可用初始状态）
init_states = torch.load(init_states_path)

# reset：重置环境（向量环境接口）
env.reset()

# 取第一个初始状态（切片 [0:1] 保留 batch 维度，匹配向量环境）
init_state = init_states[0:1]
# dones：记录每个并行环境是否已经 done（这里长度=env_num=1）
dones = [False]

# 重置算法内部状态（例如 policy 的 latent_queue、RNN hidden state 等）
algo.reset()

# set_init_state：把环境设置到指定初始状态，并返回初始观测 obs
obs = env.set_init_state(init_state)

# Make sure the gripepr is open to make it consistent with the provided demos.
# 有些示范数据一开始夹爪是打开的；这里先执行几步“零动作”让状态对齐（尤其是夹爪）
dummy_actions = np.zeros((env_num, action_dim))
for _ in range(5):
    obs, _, _, _ = env.step(dummy_actions)  # step 返回 (obs, reward, done, info)，这里先忽略后 3 个

# steps：计步器，防止无限循环
steps = 0

# obs_tensors：用来存每一帧图像，后面写视频用；每个环境一个列表
obs_tensors = [[]] * env_num
# 在最大步数以内不断 rollout（直到成功/结束）
while steps < cfg.eval.max_steps:
    steps += 1  # 每循环一次算一步
    # 把原始 obs 转成 policy 输入格式（tensor + task_emb + device 等）
    data = raw_obs_to_tensor_obs(obs, task_emb, cfg)
    # 用当前 policy 产生动作（numpy）
    action = algo.policy.get_action(data)

    # 把动作送进环境，得到下一步观测和 done 标记
    obs, reward, done, info = env.step(action)

    # 记录每个并行环境的 done 状态，并把当前帧图像存下来
    for k in range(env_num):
        dones[k] = dones[k] or done[k]  # 一旦 done 就保持 done
        obs_tensors[k].append(obs[k]["agentview_image"])  # 取 agentview_image 作为视频帧
    # 如果所有环境都 done，就提前结束 rollout
    if all(dones):
        break

# visualize video
# obs_tensor: (env_num, T, H, W, C)
# 把第 0 个环境的帧序列取出来，并做一个简单的通道/方向处理（img[::-1] 取决于数据存储顺序）
images = [img[::-1] for img in obs_tensors[0]]
# fps：视频帧率
fps = 30
# 创建视频写入器，输出到 tmp_video.mp4
writer  = imageio.get_writer('tmp_video.mp4', fps=fps)
# 逐帧写入视频
for image in images:
    writer.append_data(image)
writer.close()  # 关闭写入器，确保文件写完

# 读取生成的 mp4 二进制内容
video_data = open("tmp_video.mp4", "rb").read()
# 把 mp4 转成 base64 并拼成 <video> 标签，直接在 notebook 里播放
video_tag = f'<video controls alt="test" src="data:video/mp4;base64,{b64encode(video_data).decode()}">'
# 显示 HTML（Jupyter 会渲染成视频播放器）
HTML(data=video_tag)

